# Pretraining, Fine-Tuning & Alignment

The modern LLM training pipeline has three stages: pretraining, supervised fine-tuning (SFT), and preference alignment (RLHF/DPO). This note covers when to use each stage, implements DPO loss and LoRA, and discusses catastrophic forgetting.

## What Interviewers Test
- The pretraining → SFT → alignment pipeline and what each stage achieves
- DPO loss formula and intuition vs RLHF
- When to fine-tune vs prompt engineer vs use RAG
- LoRA: the low-rank factorization, parameter count math, and why it works
- Catastrophic forgetting and how to mitigate it
- Data quality > data quantity principle

## The Training Pipeline

```
Pretraining (next-token prediction on web-scale text)
  → Base model: knows language, facts, code, reasoning
  → Does NOT follow instructions; may generate harmful content

Supervised Fine-Tuning (SFT) on instruction-following examples
  → Instruction-tuned model: follows instructions, better format
  → May still produce harmful/unaligned outputs

Preference Alignment (RLHF / DPO) on preference pairs
  → Aligned model: follows human preferences, refuses harmful requests
  → What you get from Claude, GPT-4, LLaMA-chat
```


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
np.random.seed(42)
torch.manual_seed(42)

# ===== DPO (Direct Preference Optimization) Loss =====
def dpo_loss(pi_logprob_chosen, pi_logprob_rejected,
             ref_logprob_chosen, ref_logprob_rejected,
             beta=0.1):
    """
    DPO loss from Rafailov et al. 2023.
    Trains policy pi to prefer chosen over rejected responses
    relative to a reference policy ref.
    
    log-ratio advantage: log(pi(y_w)/ref(y_w)) - log(pi(y_l)/ref(y_l))
    Loss: -log(sigmoid(beta * advantage))
    
    beta: regularization strength (larger = closer to reference)
    """
    log_ratio_chosen   = pi_logprob_chosen   - ref_logprob_chosen
    log_ratio_rejected = pi_logprob_rejected - ref_logprob_rejected
    advantage = log_ratio_chosen - log_ratio_rejected
    return -F.logsigmoid(beta * advantage).mean()

# Simulate toy preference data
batch_size = 32
# Policy log-probs on (chosen, rejected) responses
pi_chosen   = torch.randn(batch_size) - 0.5   # slightly prefers chosen
pi_rejected = torch.randn(batch_size) - 1.0   # dislikes rejected
ref_chosen  = torch.zeros(batch_size)          # reference is neutral
ref_rejected= torch.zeros(batch_size)

loss = dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1)
print(f"DPO loss: {loss.item():.4f}")
print()
print("DPO vs RLHF:")
print("  RLHF: train reward model → optimize policy with RL (PPO)")
print("  DPO:  reparameterize reward model → cross-entropy loss directly on preference pairs")
print("  DPO advantage: no separate reward model, no RL training loop, simpler to implement")
print("  RLHF advantage: can use continuous reward signals, separate reward model for evaluation")


## Decision Table: Fine-Tune vs Prompt vs RAG

| Method | Best for | Not good for | Cost |
|---|---|---|---|
| **Prompt engineering** | Capability already in model; prototyping | Tasks needing new knowledge/format | Near-zero |
| **Few-shot** | Adapting output format; style control | Very long examples; high latency | Low (tokens) |
| **RAG** | Recent/external knowledge; source attribution | Reasoning over implicit knowledge | Medium (retrieval) |
| **LoRA fine-tune** | Style/format change; domain adaptation | Adding entirely new knowledge | Medium (training) |
| **Full fine-tune** | Catastrophic forgetting mitigation; max quality | Cost; infrastructure; data quality | High |

> 💡 **Interview Tip:** Interviewers love "fine-tune vs RAG." Frame it as: fine-tune changes what the model *knows how to do*; RAG changes what information it has access to. Fine-tune for skills; RAG for facts.


In [ ]:
# ===== LoRA from Scratch =====
class LoRALinear(nn.Module):
    """
    LoRA: Low-Rank Adaptation.
    Original weight W (d_out, d_in) is frozen.
    Add trainable delta: W + BA where B:(d_out, r), A:(r, d_in)
    
    Trainable params: r * (d_in + d_out) vs d_in * d_out for full fine-tune
    """
    def __init__(self, d_in, d_out, rank=4, alpha=8.0):
        super().__init__()
        self.W   = nn.Parameter(torch.randn(d_out, d_in) / d_in**0.5, requires_grad=False)
        self.A   = nn.Parameter(torch.randn(rank, d_in) / rank**0.5)   # trainable
        self.B   = nn.Parameter(torch.zeros(d_out, rank))               # trainable, init to 0
        self.scale = alpha / rank   # scaling factor

    def forward(self, x):
        return x @ self.W.T + self.scale * (x @ self.A.T @ self.B.T)

# Parameter count comparison
d_in, d_out, rank = 4096, 4096, 16
full_params = d_in * d_out
lora_params = rank * (d_in + d_out)
print(f"Full fine-tune params for one linear layer: {full_params:,}")
print(f"LoRA params (rank={rank}):                   {lora_params:,}")
print(f"Reduction: {full_params / lora_params:.1f}x fewer trainable parameters")
print()
# For a 7B model with ~28 attention layers, each with 4 weight matrices:
n_layers, n_matrices = 32, 4
lora_total = n_layers * n_matrices * rank * (d_in + d_out)
print(f"LoRA trainable params (full LLaMA-7B, rank={rank}): {lora_total/1e6:.1f}M")
print(f"vs 7B full fine-tune: 7,000M params")
print(f"→ {7000 / (lora_total/1e6):.0f}x reduction in trainable params")

# Quick functional test
layer = LoRALinear(64, 64, rank=4)
x_test = torch.randn(4, 64)
out = layer(x_test)
print(f"\nLoRA layer output shape: {out.shape}")


## Catastrophic Forgetting Demo


In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

# Catastrophic forgetting: fine-tuning on Task B degrades Task A performance
# (Simplified with sklearn; same principle applies to neural networks)
np.random.seed(42)

X_A, y_A = make_classification(n_samples=500, n_features=10, n_informative=6, random_state=1)
X_B, y_B = make_classification(n_samples=500, n_features=10, n_informative=6, random_state=2)

from sklearn.linear_model import LogisticRegression

# Pretrain on Task A
model = LogisticRegression(max_iter=500, warm_start=True).fit(X_A, y_A)
acc_A_before = accuracy_score(y_A, model.predict(X_A))

# Fine-tune on Task B only (simulating catastrophic forgetting)
# In a neural network, gradient updates for B can overwrite A's features
model.fit(X_B, y_B)  # forget Task A, fit only B
acc_A_after  = accuracy_score(y_A, model.predict(X_A))
acc_B_after  = accuracy_score(y_B, model.predict(X_B))

print("Catastrophic Forgetting Illustration:")
print(f"Task A accuracy BEFORE fine-tuning on B: {acc_A_before:.3f}")
print(f"Task A accuracy AFTER  fine-tuning on B: {acc_A_after:.3f}")
print(f"Task B accuracy after fine-tuning:        {acc_B_after:.3f}")
print()
print("Mitigations in practice:")
print("  - Replay: mix Task A data into Task B fine-tuning")
print("  - LoRA: freeze base weights; only train adapter (base model retains Task A)")
print("  - EWC (Elastic Weight Consolidation): regularize changes to important weights")


## Common Interview Questions

**Q: What is the difference between RLHF and DPO?**
RLHF trains a separate reward model from human preference pairs, then uses RL (PPO) to optimize the policy to maximize reward. DPO reparameterizes the RL objective to directly optimize a cross-entropy loss on preference pairs — no explicit reward model, no RL loop. DPO is simpler and often matches RLHF quality, but RLHF allows a separate, inspectable reward model and can handle online feedback.

**Q: Why does LoRA work? What is the intuition?**
The hypothesis is that fine-tuning changes reside in a low-dimensional subspace — most of the weight update matrix has low intrinsic rank. LoRA parameterizes this update as W + BA where B and A are low-rank matrices. In practice, rank 4–64 is enough to match full fine-tuning on most tasks, with 100× fewer trainable parameters.

**Q: How much training data do you need for fine-tuning?**
Quality matters more than quantity. A few thousand high-quality instruction-response pairs (cleaned, diverse, correctly formatted) typically outperform tens of thousands of noisy examples. For LoRA fine-tuning: 1K–10K examples is often sufficient for format/style adaptation. For domain adaptation: more is better, but diminishing returns after ~100K examples.

**Q: What is catastrophic forgetting and how do you mitigate it?**
When fine-tuning on a new task, gradient updates overwrite the representations learned during pretraining, causing performance to degrade on the original capabilities. Mitigations: (1) LoRA — freeze base weights entirely; (2) replay — mix pretraining data into fine-tuning; (3) EWC — add regularization to protect important weights; (4) low learning rate with early stopping.

## Key Takeaways
- Pipeline: pretraining (language) → SFT (instruction-following) → alignment (preference)
- DPO loss: -log sigmoid(β × (log ratio chosen - log ratio rejected)); simpler than RLHF, no RL
- Fine-tune for skills; RAG for facts — this is the key decision framework
- LoRA: W → W + BA; rank-r adaptation; 100× fewer trainable params; freeze base weights
- LoRA prevents catastrophic forgetting: base model weights don't change
- Data quality >> data quantity for fine-tuning; 1K–10K clean examples often enough